# Strata Cache Builder — DevZoneX

Prepare the datasets that let a later **T4 ×2** session start Strata in a few
minutes with almost no local disk use.

## What this notebook is for

| Artifact | Dataset | Where it is created |
|---|---|---|
| Compiled runtime (~1.3 GiB) | `strata-runtime-cache` | any session that can install Strata |
| MTP draft layer (~5.2 GiB) | `strata-mtp-rt` | a session where Strata actually ran |
| Model weights + packs (~80 GiB) | `strata-model-<family>-<variant>` | a session where Strata actually ran |

Then the launcher reads all three in place: local disk holds only the extracted
runtime, and the model is **never copied again**.

## When to enable the GPU — read this twice

| Phase | Accelerator | Why |
|---|---|---|
| 1. Load helpers, restore runtime, probe `strata` | **None (CPU)** | the runtime restore and `--help` need no CUDA |
| 2. Review the plan / probe download | **None (CPU)** | cheap, and tells you the real file list and sizes |
| 3. First model launch and dataset creation | **T4 ×2 required** | the engine is compiled for the GPU it runs on (`sm_75`), and the model is fetched by the launch path |
| 4. Publish datasets that already exist (e.g. an attached raw bundle) | **None (CPU)** | upload is file I/O via the Kaggle CLI |
| 5. Every later serving session | **T4 ×2** | use `Strata_Kaggle_Launcher.ipynb`, not this notebook |

**Changing the accelerator restarts the Kaggle kernel and wipes `/root`.** Attach
the GPU *before* creating files you intend to upload, and run the upload cell in
the same session in which those files exist.

**Whether the model can be fetched on CPU depends on your Strata build.** Section
3 probes for a fetch/prepare command. If none exists, section 3c attempts a
download-only `strata launch` on CPU: Strata's terminal documentation says a
launch downloads the model even when no engine exists for the computer, then
ends without serving. If that works, the whole data phase stays on CPU; if the
files do not appear, one GPU session is unavoidable, and the notebook says so
instead of guessing.


## 0. Configuration

Edit this cell only. The same values must be used by `Strata_Kaggle_Launcher.ipynb`
later, otherwise the datasets will not match the model you launch.


In [ ]:
# ===== USER CONFIGURATION: EDIT THIS CELL =====
KAGGLE_USERNAME = "your-kaggle-username"

# Must match the launcher that will use these datasets.
MODEL_FAMILY = "gyro"
MODEL_VARIANT = "gyro-s"
CONTEXT = "32k"
VISION = "off"
DRAFTING = "auto"                      # "auto"/"mtp" = MTP draft layer; "dflash2" = Qwen3.8-Flash-Next only; "off" = none

MODEL_DATA_DIR = "/root/strata-local"  # everything large lives here
MODEL_RESTORE_MODE = "symlink"         # default: read from attached datasets, do not copy

# One dataset per artifact. The model slug is derived from family/variant.
RUNTIME_DATASET_SLUG = "strata-runtime-cache"
DRAFT_LAYER_DATASET_SLUG = "strata-mtp-rt"
MODEL_DATASET_SLUG = "strata-model-" + f"{MODEL_FAMILY}-{MODEL_VARIANT}".lower()

# Optional external source for the draft layer. Blank = your own
# <KAGGLE_USERNAME>/strata-mtp-rt when attached. The public
# "clothespin/strata-qfn-assets-lite" mirror is opt-in only.
DRAFT_LAYER_DATASET = ""

# Optional raw Strata file bundle to adopt instead of downloading (Dense/PLE
# GGUFs, experts.bin, a profile). Only use it if the launch plan lists the same
# file names and sizes; give "owner/slug" or leave blank.
MODEL_FILES_DATASET = ""

# Upload switches. The model upload additionally requires the license line below.
UPLOAD_RUNTIME_CACHE = True            # ~1.3 GiB
UPLOAD_DRAFT_LAYER = True              # ~5.2 GiB, shared by every variant
UPLOAD_MODEL_CACHE = True              # ~80 GiB; keep True only if disk/time allow
I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE = False
FORCE_UPDATE_RUNTIME_CACHE = False
FORCE_UPDATE_DRAFT_LAYER = False
FORCE_UPDATE_MODEL_CACHE = False

# Probe settings.
PROBE_TIMEOUT_SECONDS = 300

print("Model:", MODEL_FAMILY, MODEL_VARIANT)
print("Model dataset:", f"{KAGGLE_USERNAME}/{MODEL_DATASET_SLUG}")
print("Runtime dataset:", f"{KAGGLE_USERNAME}/{RUNTIME_DATASET_SLUG}")
print("Draft-layer dataset:", DRAFT_LAYER_DATASET or f"{KAGGLE_USERNAME}/{DRAFT_LAYER_DATASET_SLUG}")
print("Data dir:", MODEL_DATA_DIR, "| restore mode:", MODEL_RESTORE_MODE)
print("Uploads runtime/draft/model:", UPLOAD_RUNTIME_CACHE, UPLOAD_DRAFT_LAYER, UPLOAD_MODEL_CACHE)
print("Model license confirmed:", I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE)


## 1. Load helpers — CPU-safe

This clones the public repository, reads the Kaggle token from Secrets, and
reports hardware **without requiring a GPU**. If a `scripts` package import ever
fails in a scratch kernel, the importlib fallback below loads the helper file
directly.


In [ ]:
import importlib.util, json, os, shutil, subprocess, sys
from pathlib import Path

REPO_URL = "https://github.com/vinodhkumark/devzonex-strata-kaggle.git"
REPO_DIR = Path("/kaggle/working/devzonex-strata-kaggle")
HELPER = REPO_DIR / "scripts" / "strata_kaggle.py"

REQUIRED_HELPERS = (
    "validate_config", "environment_report", "restore_runtime_cache",
    "restore_model_cache", "restore_draft_layer", "cache_status",
    "run_plan", "upload_runtime_cache", "upload_model_cache", "upload_draft_layer",
)


def helper_is_current() -> bool:
    try:
        text = HELPER.read_text(encoding="utf-8")
    except OSError:
        return False
    return all(("def " + name) in text for name in REQUIRED_HELPERS)


if not helper_is_current():
    if REPO_DIR.exists():
        shutil.rmtree(REPO_DIR)
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)

spec = importlib.util.spec_from_file_location("strata_kaggle", HELPER)
sk = importlib.util.module_from_spec(spec)
sys.modules["strata_kaggle"] = sk
spec.loader.exec_module(sk)

MISSING = [name for name in REQUIRED_HELPERS if not hasattr(sk, name)]
if MISSING:
    raise RuntimeError("The helper checkout is missing: " + ", ".join(MISSING))

CONFIG = sk.validate_config(
    MODEL_FAMILY, MODEL_VARIANT, CONTEXT, DRAFTING, VISION, MODEL_RESTORE_MODE, "",
)
KAGGLE_ENV = sk.kaggle_env(KAGGLE_USERNAME, secret_name="KAGGLE_API_TOKEN")
HARDWARE = sk.hardware_summary()
print("GPU(s):", HARDWARE["gpu_names"] or "none (CPU session)")
print("Helper:", sk.__file__)


## 2. Restore the runtime and inspect what is attached — CPU-safe

The runtime is the one artifact that must be **extracted locally**: it is an
executable tree (its own Python 3.12, engine binaries, a mutable `terminal/current`),
so a read-only input mount cannot host it.

Attach `strata-runtime-cache` under **Add Input → Your Datasets**. On a returning
session, attaching the model and draft-layer datasets too lets the next cell show
exactly what the launcher will link.


In [ ]:
MODEL_ROOT = Path(MODEL_DATA_DIR)
MODEL_ROOT.mkdir(parents=True, exist_ok=True)

ENVIRONMENT = sk.environment_report(require_cuda=False, path=MODEL_DATA_DIR)
RUNTIME_CACHE_RESTORED = sk.restore_runtime_cache(KAGGLE_USERNAME, RUNTIME_DATASET_SLUG)
MODEL_CACHE_RESTORED = sk.restore_model_cache(
    KAGGLE_USERNAME, MODEL_DATASET_SLUG, MODEL_ROOT, workers=8,
    family=MODEL_FAMILY, variant=MODEL_VARIANT, restore_mode=MODEL_RESTORE_MODE,
)

_draft_source = DRAFT_LAYER_DATASET.strip() or (KAGGLE_USERNAME + "/" + DRAFT_LAYER_DATASET_SLUG)
_draft_owner, _, _draft_slug = _draft_source.partition("/")
DRAFT_LAYER_RESTORED = sk.restore_draft_layer(
    _draft_owner.strip(), _draft_slug.strip(), MODEL_ROOT, restore_mode=MODEL_RESTORE_MODE,
)

sk.install_strata_if_missing()
CACHE_STATUS = sk.cache_status(MODEL_ROOT)

print("Runtime cache restored:", RUNTIME_CACHE_RESTORED)
print("Model cache restored:", MODEL_CACHE_RESTORED)
print("Draft layer restored:", DRAFT_LAYER_RESTORED)
print("Restore mode:", MODEL_RESTORE_MODE)
print("Strata executable:", shutil.which("strata") or "(not on PATH yet)")


## 3. Probe Strata — CPU-safe

This is the cell that answers "can the data phase run without a GPU?". It runs
`strata --help`, `strata launch --help`, and the launch plan, then writes the
whole transcript to `/kaggle/working/strata-probe.txt`.

The plan is cheap and runs before any large download. It is the only reliable
source for the model's real file list and sizes.


In [ ]:
STRATA = shutil.which("strata")
if not STRATA:
    raise RuntimeError(
        "strata is not installed. Either attach strata-runtime-cache and rerun the "
        "runtime cell, or accept that this notebook cannot probe a fresh environment."
    )

transcript = []


def record(title, result):
    block = ("### " + title + "\nexit: " + str(result.returncode)
             + "\n--- stdout ---\n" + (result.stdout or "")
             + "\n--- stderr ---\n" + (result.stderr or "") + "\n")
    transcript.append(block)
    print(block)


for title, cmd in (("strata --help", [STRATA, "--help"]),
                   ("strata launch --help", [STRATA, "launch", "--help"])):
    record(title, subprocess.run(cmd, text=True, capture_output=True,
                                 timeout=PROBE_TIMEOUT_SECONDS, check=False))

try:
    PLAN_OUTPUT = sk.run_plan(
        family=MODEL_FAMILY, variant=MODEL_VARIANT, context=CONTEXT, vision=VISION,
        gpu_list="0", data_dir=MODEL_ROOT, drafting=DRAFTING,
    )
    PLAN_OK = True
except Exception as exc:
    PLAN_OUTPUT = "PLAN FAILED: " + str(exc)
    PLAN_OK = False

print(PLAN_OUTPUT)
transcript.append("### plan (ok=" + str(PLAN_OK) + ")\n" + PLAN_OUTPUT + "\n")

PROBE_PATH = Path("/kaggle/working/strata-probe.txt")
PROBE_PATH.write_text("\n".join(transcript), encoding="utf-8")
print("Transcript written to", PROBE_PATH)


## 3b. Read the probe result

`CPU_FETCH_HINTS` are words that would indicate a fetch/prepare step that does not
need CUDA. A hit is a lead, not proof — confirm the subcommand actually downloads
the model files by reading its help output.


In [ ]:
import re

# Separate the recorded blocks so prose in one help page cannot be mistaken for
# a command in another.
BLOCKS = {}
for block in "\n".join(transcript).split("### ")[1:]:
    title, _, body = block.partition("\n")
    BLOCKS[title.strip()] = body

ROOT_HELP = BLOCKS.get("strata --help", "")
LAUNCH_HELP = BLOCKS.get("strata launch --help", "")

# A subcommand line is indented and starts a column of its own.
SUBCOMMAND = re.compile(r"^\s{2,}([a-z][a-z0-9-]+)\s{2,}\S")
SUBCOMMANDS = sorted({
    match.group(1)
    for match in (SUBCOMMAND.match(line) for line in ROOT_HELP.splitlines())
    if match
})
FLAGS = sorted(set(re.findall(r"--[a-z][a-z0-9-]+", LAUNCH_HELP)))

print("Subcommands found under 'strata':", SUBCOMMANDS or "none parsed")
print("Flags found under 'strata launch':", FLAGS or "none parsed")

# Only report a hint when the word is an actual command or flag, not a sentence.
CPU_FETCH_WORDS = ("fetch", "download", "prepare", "prefetch", "pull", "warm")
COMMAND_HITS = [w for w in CPU_FETCH_WORDS
                if any(w == s or w in s.split("-") for s in SUBCOMMANDS)]
FLAG_HITS = [w for w in CPU_FETCH_WORDS if any(w in f for f in FLAGS)]
print("CPU fetch candidates as subcommands:", COMMAND_HITS or "none")
print("CPU fetch candidates as flags:", FLAG_HITS or "none")
print("Plan produced output:", PLAN_OK)

if not PLAN_OK:
    print("\n=> One GPU session is required: the plan itself needs a usable GPU session.")
elif COMMAND_HITS:
    print("\n=> Run '<command> --help' for each candidate above. A real fetcher downloads")
    print("   model files without CUDA; upload them from the same CPU session.")
elif FLAG_HITS:
    print("\n=> A launch flag may limit the run to downloading. Read its help text, then")
    print("   use a download-only launch here on CPU instead of the GPU section.")
else:
    print("\n=> 'download'/'fetch' only appeared in descriptions, not as commands or flags.")
    print("   The model arrives through 'strata launch'. Strata is documented to download")
    print("   the model even when no engine exists for the computer, so a CPU probe launch")
    print("   may still populate the files - see the next cell before committing to a GPU.")

PLAN_TEXT = BLOCKS.get("plan (ok=" + str(PLAN_OK) + ")", PLAN_OUTPUT)
SIZE_HITS = re.findall(r"\b\d+(?:\.\d+)?\s*(?:GiB|GB|MiB|MB)\b", PLAN_TEXT)
print("\nSizes mentioned by the plan:", SIZE_HITS[:20] or "none parsed")
print("Model files mentioned by the plan:",
      sorted(set(re.findall(r"[A-Za-z0-9._/-]+\.(?:gguf|bin)", PLAN_TEXT)))[:20] or "none parsed")
print("\nPaste this whole cell's output plus the 'strata launch --help' block before")
print("trusting any size or filename.")


## 3c. Download the model on CPU, without serving it

Strata's own documentation says it plainly: when there is no engine for the
computer, `strata launch` **does not refuse** — it downloads the model and ends
with a message naming the problem. The plan above confirms that shape: it names
the downloads and reports that engine `0.1.55` did not pass the check on this CPU
session.

So this cell runs a normal launch on CPU. Expect it to download the model and the
draft layer, prepare the local files, then stop without serving. That message is
the success condition for this step, not a failure. After it finishes, check that
`models/`, `packs/` and `mtp/` have real bytes, then upload them in section 5 —
no GPU ever attached.

Budget from the plan: **~63.7 GB** (model 58.5 GB + draft layer 5.2 GB). Check
free space on `MODEL_DATA_DIR` first, and remember preparation can add pack files
on top of the download. If space is tight, `--mtp off` saves 5.2 GB but you then
cannot create `strata-mtp-rt`.

Set `CPU_DOWNLOAD_ONLY = True` only after reading the plan. The run can take
hours; it is resumable, so an interrupted session does not start over.


In [ ]:
CPU_DOWNLOAD_ONLY = False   # set True only after the plan lists missing files

if not CPU_DOWNLOAD_ONLY:
    print("Skipped. Set CPU_DOWNLOAD_ONLY = True to download the model on this CPU session.")
else:
    free_gb = shutil.disk_usage(MODEL_ROOT).free / (1024 ** 3)
    print(f"Free space on {MODEL_DATA_DIR}: {free_gb:.1f} GiB")
    print("Plan estimate was about 63.7 GB of downloads (58.5 model + 5.2 draft),")
    print("plus any prepared pack files. Stop now if that does not fit:")

    download_args = [
        "strata", "launch",
        "--family", MODEL_FAMILY, "--model", MODEL_VARIANT,
        "--data-dir", MODEL_DATA_DIR, "--context", CONTEXT, "--vision", VISION,
        "--mtp", "on", "--drafting", DRAFTING, "--tune", "skip",
        "--access", "local", "--yes", "--no-tui",
    ]
    print("\nRunning:", " ".join(download_args))
    print("A missing/unsuitable-engine message at the end is expected on a CPU session.")
    result = subprocess.run(download_args, text=True, capture_output=True,
                            timeout=12 * 60 * 60, check=False)
    print("exit:", result.returncode)
    print(result.stdout[-12000:])
    print("STDERR:", result.stderr[-4000:])

    print("\nFiles now under", MODEL_DATA_DIR)
    sk.cache_status(MODEL_ROOT)
    print("\nUpload them in section 5 if models/packs/mtp have real bytes.")
    print("If they are still empty, the CPU download path does not work in this build")
    print("and section 4 (T4 x2) is required.")


## 4. GPU gate — attach T4 ×2 **now** if section 3 says you must

Only run this cell after switching the accelerator to **GPU T4 ×2** (or another
supported GPU) in the notebook settings. Kaggle restarts the kernel when the
accelerator changes, so after switching you must rerun sections 1–2 before this
one. Section 3's probe does not need to be repeated unless you want the plan
re-checked on the GPU.

This cell refuses to continue on a CPU session instead of failing later during a
10–15 minute compile.


In [ ]:
GPU_REPORT = sk.environment_report(require_cuda=True, path=MODEL_DATA_DIR)
DRAFTING_STATUS = sk.drafting_status(MODEL_ROOT, DRAFTING)
print("Drafting status:", json.dumps(DRAFTING_STATUS, indent=2))

free = GPU_REPORT["disk_free_gb"]
print("Free disk on", GPU_REPORT["disk_path"], ":", free, "GiB")
print("Local disk should hold only the runtime. If models/packs show real bytes below,")
print("an earlier 'copy' restore is still occupying space.")
sk.cache_status(MODEL_ROOT)


## 5. Create or refresh the datasets

Run this in the session where the files exist. Order is intentional: runtime
first (cheap, always useful), then the draft layer, then the ~80 GiB model last.

**On a CPU session** this only works for artifacts that already exist — for
example an adopted raw bundle, or datasets you re-publish. It cannot create model
files that were never downloaded.

Wait until every dataset shows **Ready** on its Kaggle page before ending the
session. Uploading is the step that pays GPU wall-clock if the accelerator is
attached; there is no way around that for files only a GPU session can produce.


In [ ]:
MODEL_FILES_PATH = None
if MODEL_FILES_DATASET.strip():
    _bundle_owner, _, _bundle_slug = MODEL_FILES_DATASET.partition("/")
    MODEL_FILES_PATH = sk.adopt_model_bundle(
        _bundle_owner.strip() or KAGGLE_USERNAME, _bundle_slug.strip(), MODEL_ROOT,
        MODEL_FAMILY, MODEL_VARIANT, restore_mode=MODEL_RESTORE_MODE,
    )
    print("Model files adopted from dataset:", MODEL_FILES_PATH)

RUNTIME_UPLOADED = DRAFT_LAYER_UPLOADED = MODEL_UPLOADED = False

if UPLOAD_RUNTIME_CACHE:
    RUNTIME_UPLOADED = sk.upload_runtime_cache(
        username=KAGGLE_USERNAME, slug=RUNTIME_DATASET_SLUG,
        env=KAGGLE_ENV, force=FORCE_UPDATE_RUNTIME_CACHE,
    )
    print("Runtime cache upload submitted:", RUNTIME_UPLOADED)

if UPLOAD_DRAFT_LAYER:
    DRAFT_LAYER_UPLOADED = sk.upload_draft_layer(
        data_dir=MODEL_ROOT, username=KAGGLE_USERNAME,
        slug=DRAFT_LAYER_DATASET_SLUG, env=KAGGLE_ENV,
        force=FORCE_UPDATE_DRAFT_LAYER,
    )
    print("Draft-layer upload submitted:", DRAFT_LAYER_UPLOADED)

if UPLOAD_MODEL_CACHE:
    MODEL_UPLOADED = sk.upload_model_cache(
        data_dir=MODEL_ROOT, username=KAGGLE_USERNAME, slug=MODEL_DATASET_SLUG,
        family=MODEL_FAMILY, variant=MODEL_VARIANT, env=KAGGLE_ENV,
        license_confirmed=I_CONFIRM_MODEL_LICENSE_AND_PRIVATE_STORAGE,
        force=FORCE_UPDATE_MODEL_CACHE,
    )
    print("Model cache upload submitted:", MODEL_UPLOADED)

print("\nDatasets:")
for name in (RUNTIME_DATASET_SLUG, DRAFT_LAYER_DATASET_SLUG, MODEL_DATASET_SLUG):
    print(" -", KAGGLE_USERNAME + "/" + name)


## 6. Verify the datasets exist and are yours

This uses the Kaggle CLI that section 1 installed. It prints status for the three
slugs; the model may be absent until section 5 has run to completion.


In [ ]:
for slug in (RUNTIME_DATASET_SLUG, DRAFT_LAYER_DATASET_SLUG, MODEL_DATASET_SLUG):
    print("=" * 20, slug)
    result = subprocess.run(
        ["kaggle", "datasets", "status", KAGGLE_USERNAME + "/" + slug],
        text=True, capture_output=True, env=KAGGLE_ENV, check=False,
    )
    print((result.stdout or result.stderr or "").strip()[:2000])


## 7. Hand off to the launcher

When all three datasets are **Ready**, next session is:

1. Open `Strata_Kaggle_Launcher.ipynb` and switch the accelerator to **T4 ×2**.
2. Attach under **Add Input → Your Datasets**:
   - `strata-runtime-cache`
   - `strata-mtp-rt`
   - `strata-model-<family>-<variant>`
3. Keep the launcher's defaults: `MODEL_RESTORE_MODE = "symlink"`, and
   `DRAFT_LAYER_DATASET = ""` (which resolves to your own `strata-mtp-rt`).
4. Run the launcher cells in order. Expected startup: runtime restore ~1 min,
   model/packs/draft linked rather than copied, then the plan and launch.

Expected local disk: roughly **1.5 GiB**, not ~80 GiB.

Measure the symlink trade-off once with the launcher's optional quality gate. If
decode tokens/s are unacceptable, the middle ground is copying only `models/`
while symlinking packs and the draft layer — but that needs the free disk this
design is avoiding.

## If something goes wrong

| Symptom | Meaning | Fix |
|---|---|---|
| `restore_model_cache` says "not attached" | dataset missing from Add Input | attach it, rerun section 2 |
| restore says "already existed as local copies and were kept" | an earlier `copy` restore left files | delete `MODEL_DATA_DIR/models`, rerun in `symlink` mode |
| plan fails on CPU with a CUDA error | no CPU-only planning in this build | run section 4 on T4 ×2 |
| upload exits without creating a dataset | license flag is False, or no files under `models/`/`packs/` | confirm the license, check the directory |
| the model dataset is larger than Kaggle's free disk | ~80 GiB may not fit next to runtime | keep the model dataset attached and never copy it; do not use `copy` mode |
